# 📘 Notebook 08: Bag of Words, TF-IDF and Search

### Course: *From Words to Meaning: Natural Language Processing in Python*
**Instructor:** Ioannis Tsioulis

*Module C: Counting Words · Notebook 2 of 3 in this module · (8 of 18 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Represent a document as a **bag-of-words** vector and a collection as a **document-term matrix**
- Compute the **cosine similarity** between two documents by hand and explain why it is preferred to distance
- Explain the idea behind **TF-IDF** and compute it by hand
- Use scikit-learn's `CountVectorizer` and `TfidfVectorizer`, and say what a **sparse matrix** is
- Build a working **search engine** over thousands of news articles in a few lines
- Measure the quality of search results, and state what this representation cannot capture

> **Prerequisites:** Notebooks 01 and 03. Vectors and the dot product, as covered in *From Python to Deep Learning & Fine-Tuning*, Notebook 05 (NumPy).
>
> 🔭 **Why this notebook matters:** the previous notebook used counts to model the order of words. This one throws the order away on purpose, and gains something powerful in exchange: every document becomes a point in space, and questions about text become questions about geometry. *"Which documents are about the same thing?"* becomes *"which points are close together?"*. That single move is behind search engines, recommendation systems and document classifiers, and it prepares the ground for everything in Module D.

> ℹ️ **Setup note.** Run this cell once. It downloads a collection of news articles.

In [ ]:
import nltk
for package in ["reuters", "gutenberg", "punkt", "punkt_tab"]:
    nltk.download(package, quiet=True)

import math
import numpy as np
from collections import Counter

## 1. Documents as vectors

### Intuition first
Imagine tipping all the words of a document into a bag and shaking it. The order is gone. What remains is which words are in the bag and how many of each. It sounds like vandalism, and yet from the bag alone you could still tell a football report from a recipe.

Now fix a list of all the words we care about, the vocabulary, in some agreed order. A document can then be written as a list of numbers: how many times the first vocabulary word occurs in it, how many times the second, and so on. That list is a **vector**.

### Formal definition
Given a vocabulary of $V$ words, the **bag-of-words** representation of a document is a vector of length $V$ whose $i$-th entry is the number of times the $i$-th word occurs in the document. Stacking the vectors of all documents gives the **document-term matrix**: one row per document, one column per word.

In [ ]:
documents = [
    "the cat sat on the mat",
    "the dog sat on the log",
    "the cat chased the dog",
    "stock markets fell sharply today",
]

vocabulary = sorted(set(word for document in documents for word in document.split()))
print(len(vocabulary), "words:", vocabulary)

def count_vector(text):
    counts = Counter(text.split())
    return [counts[word] for word in vocabulary]

vectors = [count_vector(document) for document in documents]

print()
print(" " * 34 + " ".join(f"{word[:4]:>4}" for word in vocabulary))
for document, vector in zip(documents, vectors):
    print(f"{document:<34}" + " ".join(f"{value:>4}" for value in vector))

Each row is a document and each column a word. Most entries are zero, because any one document uses only a few of the words in the vocabulary. Remember that observation. It becomes important when the vocabulary has tens of thousands of words.

## 2. Comparing documents: cosine similarity

### Intuition first
Two documents about the same subject use many of the same words, so their vectors have large values in the same positions. Geometrically, they **point in a similar direction**.

Why direction, and not the distance between the points? Consider a short article and a long article on the same topic. The long one has bigger counts everywhere, so its point lies far out from the origin, far from the short one. But both point the same way. Direction captures *what a text is about*. Length only captures *how much of it there is*.

### Formal definition
The **cosine similarity** of two vectors is the cosine of the angle between them:

$$\cos(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{\lVert \mathbf{a} \rVert \, \lVert \mathbf{b} \rVert} = \frac{\sum_i a_i b_i}{\sqrt{\sum_i a_i^2}\,\sqrt{\sum_i b_i^2}}$$

For count vectors it ranges from 0 (no word in common) to 1 (same direction). The numerator is the **dot product**, the same operation that drives every neural network in the companion course.

In [ ]:
def dot(a, b):
    total = 0
    for x, y in zip(a, b):
        total += x * y
    return total

def cosine(a, b):
    return dot(a, b) / (math.sqrt(dot(a, a)) * math.sqrt(dot(b, b)))

print("Similarity between every pair of documents:\n")
for i in range(len(documents)):
    print("  ".join(f"{cosine(vectors[i], vectors[j]):.2f}" for j in range(len(documents))), "  ", documents[i])

Each document has similarity 1 with itself (the diagonal). The three sentences about animals are similar to one another, and the sentence about stock markets has similarity 0 with all of them, since it shares no word with them.

Now the claim about length. Let us paste a document to itself, so that it is twice as long and says nothing new.

In [ ]:
short = count_vector("the cat sat on the mat")
long = count_vector("the cat sat on the mat the cat sat on the mat")

distance = math.sqrt(sum((x - y) ** 2 for x, y in zip(short, long)))

print("Euclidean distance:", round(distance, 2))
print("Cosine similarity: ", round(cosine(short, long), 2))

By distance the two are far apart. By cosine they are identical, which is the answer we want.

## 3. Not all words are equally informative: TF-IDF

### Intuition first
Look again at the similarity table. *"the cat sat on the mat"* and *"the cat chased the dog"* come out as fairly similar, and a good part of that is due to the word *the*, which occurs twice in each. But *the* occurs almost everywhere. Sharing it tells us nothing.

A word is a good clue to what a document is about when two things are true:

1. it occurs **often in this document**, and
2. it occurs in **few other documents**.

*the* passes the first test and fails the second. *cat* passes both.

### Formal definition
**TF-IDF** multiplies two quantities for each word $w$ in each document $d$:

- **Term frequency** $\text{tf}(w, d)$: the number of times $w$ occurs in $d$.
- **Inverse document frequency**: $\text{idf}(w) = \log \dfrac{N}{\text{df}(w)}$, where $N$ is the number of documents and $\text{df}(w)$ is the number of documents that contain $w$.

$$\text{tf-idf}(w, d) = \text{tf}(w, d) \times \text{idf}(w)$$

A word that appears in every document has $\text{idf} = \log 1 = 0$ and vanishes. A word that appears in one document out of a thousand gets a large weight. The logarithm stops rare words from dominating too violently.

In [ ]:
N = len(documents)

document_frequency = Counter()
for document in documents:
    for word in set(document.split()):          # set: count each word once per document
        document_frequency[word] += 1

idf = {word: math.log(N / document_frequency[word]) for word in vocabulary}

print(f"{'word':<8} {'in how many documents':>22} {'idf':>6}")
for word in ["the", "sat", "cat", "dog", "mat", "markets"]:
    print(f"{word:<8} {document_frequency[word]:>22} {idf[word]:>6.2f}")

In [ ]:
def tfidf_vector(text):
    counts = Counter(text.split())
    return [counts[word] * idf[word] for word in vocabulary]

tfidf_vectors = [tfidf_vector(document) for document in documents]

print(f"{'':<26} {'counts':>7} {'tf-idf':>7}")
for i, j in [(0, 1), (0, 2), (1, 2)]:
    label = f"document {i + 1} and document {j + 1}"
    print(f"{label:<26} {cosine(vectors[i], vectors[j]):>7.2f} {cosine(tfidf_vectors[i], tfidf_vectors[j]):>7.2f}")

With TF-IDF all three similarities fall, because the contribution of *the* has been cut down. What remains reflects the words that carry content: documents 1 and 2 share *sat* and *on*, documents 1 and 3 share *cat*, and documents 2 and 3 share *dog*.

> 🧠 TF-IDF is a soft, automatic version of the stop-word list from Notebook 03. Nobody has to decide in advance which words are uninformative. The collection itself reveals them, and the answer adapts: in a collection of medical notes the word *patient* will receive a low weight without anyone asking.

## 4. The same thing with scikit-learn

Having built both representations by hand, we can hand the work to a library. scikit-learn's vectorisers tokenise the text, build the vocabulary and produce the matrix in one step.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

counter = CountVectorizer()
count_matrix = counter.fit_transform(documents)

print(counter.get_feature_names_out())
print(count_matrix.toarray())

In [ ]:
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(documents)

print(type(tfidf_matrix))
print(tfidf_matrix.shape)
print(np.round(tfidf_matrix.toarray()[0], 2))

Three details differ from our version, and it is worth knowing them:

- scikit-learn's formula for idf is slightly different (it adds 1 in a couple of places) so that no word ever gets weight exactly zero.
- Each row is **normalised to length 1**. The cosine similarity of two rows is then simply their dot product, which makes comparing documents very fast.
- The result is a **sparse matrix**: only the entries that are not zero are stored. With four tiny documents this makes no difference. With real data it is the difference between possible and impossible, as we are about to see.

## 5. A search engine over 10,000 news articles

The Reuters corpus is a collection of short financial news articles from 1987. Each has an upper-case headline on its first line.

In [ ]:
from nltk.corpus import reuters

file_ids = reuters.fileids()
articles = [reuters.raw(file_id) for file_id in file_ids]

print(len(articles), "articles\n")
print(articles[5][:400])

In [ ]:
vectoriser = TfidfVectorizer(stop_words="english", min_df=2, sublinear_tf=True, token_pattern=r"[A-Za-z]{2,}")
matrix = vectoriser.fit_transform(articles)

rows, columns = matrix.shape
print(f"Matrix: {rows:,} documents x {columns:,} words")
print(f"Total entries:    {rows * columns:,}")
print(f"Entries not zero: {matrix.nnz:,} ({matrix.nnz / (rows * columns):.2%})")

The matrix has well over a hundred million entries, and only a few in every thousand are different from zero. Stored in full it would need more than a gigabyte. Stored sparsely it takes a few megabytes.

Four options were used. `stop_words="english"` drops stop words. `min_df=2` ignores words that appear in only one article, most of which are typing errors and odd codes. `sublinear_tf=True` uses $1 + \log(\text{tf})$ in place of the raw count, so that a word used twenty times counts for more than a word used once, but not twenty times more. And `token_pattern` is a regular expression from Notebook 02: a token is two or more letters, which leaves out the many numbers in financial news.

### Searching
A query is just a very short document. We turn it into a vector **with the same vectoriser**, compute its cosine similarity with every article, and return the best matches. Because all rows have length 1, one matrix product gives all the similarities at once.

In [ ]:
def headline(text):
    return text.split("\n")[0].strip()[:70]

def search(query, how_many=5):
    query_vector = vectoriser.transform([query])
    scores = (matrix @ query_vector.T).toarray().ravel()
    best = scores.argsort()[::-1][:how_many]
    return [(scores[i], i) for i in best]

for query in ["oil prices rise after OPEC meeting", "coffee harvest in Brazil", "interest rates Bank of England"]:
    print(f'Query: "{query}"')
    for score, i in search(query):
        print(f"   {score:.2f}  {headline(articles[i])}")
    print()

That is a functioning search engine. No rules about oil or coffee were written. The results come from the geometry alone: the query and the retrieved articles point in similar directions in a space with one dimension per word.

### What is an article about?
The same vectors tell us which words characterise a document: the ones with the highest TF-IDF weight in its row.

In [ ]:
words = vectoriser.get_feature_names_out()

def keywords(i, how_many=8):
    row = matrix[i].toarray().ravel()
    best = row.argsort()[::-1][:how_many]
    return [words[j] for j in best]

for query in ["coffee harvest in Brazil", "oil prices rise after OPEC meeting", "strike at the port"]:
    score, i = search(query, how_many=1)[0]
    print(headline(articles[i]))
    print("   ", keywords(i))

In Notebook 03, raw frequency gave poor keywords, because the most frequent words of a text are frequent in every text. TF-IDF fixes exactly that: it rewards the words that are frequent **here and rare elsewhere**.

> ⚠️ **Common pitfalls**
>
> - Calling `fit_transform` on the query. The query must be mapped into the space that already exists, with `transform`. Fitting again would build a different vocabulary, and the vectors would no longer be comparable.
> - Calling `.toarray()` on a large sparse matrix. It converts the matrix to its full form and can fill the memory of your machine at once. Keep it sparse, and convert only single rows.
> - Preprocessing the documents one way and the queries another. If the documents were lower-cased and stemmed, the queries must be too.

## 6. What the bag of words cannot see

Type a query whose words differ from those of the articles, though the meaning is the same.

In [ ]:
pairs = [
    ("the car is fast", "the automobile is quick"),
    ("the car is fast", "the car is not fast"),
    ("the dog bit the man", "the man bit the dog"),
]

small = TfidfVectorizer()
for first, second in pairs:
    both = small.fit_transform([first, second])
    similarity = (both[0] @ both[1].T).toarray()[0, 0]
    print(f"{similarity:.2f}   {first!r:<24} {second!r}")

Three failures, each pointing to a later part of the course.

1. *car* and *automobile*, *fast* and *quick* are different columns of the matrix, as unrelated as *car* and *banana*. Two sentences with the same meaning look almost unrelated. **Module D** gives words vectors of their own, so that similar words are close together.
2. One added word, *not*, reverses the meaning and barely changes the vector.
3. The two sentences with opposite meanings are **identical** as bags of words. Order is gone. **Module E** brings it back.

| | Word counts | N-gram model | Bag of words + TF-IDF |
|---|---|---|---|
| Turns text into numbers | yes | yes | yes, one vector per document |
| Lets us compare documents | no | no | yes |
| Uses word order | no | a few words back | no |
| Knows that two words are similar | no | no | no |

---
## ✏️ Exercises

### Exercise 1 (Vectors by hand)
Using the `vocabulary` and the functions of Sections 1 and 2, compute the count vector of the new sentence `"the dog sat on the mat"` and its cosine similarity with each of the four documents. Which document is it closest to, and is that what you expected?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
new_vector = count_vector("the dog sat on the mat")
print(new_vector)
for document, vector in zip(documents, vectors):
    print(f"{cosine(new_vector, vector):.2f}  {document}")
```

*It is equally close to the first two documents: it shares every word but one with each of them. It is a mixture of the two, and the geometry says so.*
</details>

### Exercise 2 (Common and rare)
The fitted `vectoriser` stores the idf of every word in `vectoriser.idf_`, in the same order as `words`. Print the ten words with the **lowest** idf and ten words with the **highest**. What kind of word is at each end?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
order = vectoriser.idf_.argsort()

print("Lowest idf: ", [words[i] for i in order[:10]])
print("Highest idf:", [words[i] for i in order[-10:]])
```

*The lowest values belong to words found in a large share of the articles: "said", "mln", "dlrs", "year". They are the stop words of financial news, and no general stop-word list contains them. The highest values belong to words that occur in just two articles: rare words and names.*
</details>

### Exercise 3 (More like this)
Write `similar_articles(i, how_many=5)` that returns the articles most similar to article number `i`, leaving out the article itself. Try it on article 5 and print the headlines.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
def similar_articles(i, how_many=5):
    scores = (matrix @ matrix[i].T).toarray().ravel()
    scores[i] = -1                                   # leave out the article itself
    best = scores.argsort()[::-1][:how_many]
    return [(scores[j], j) for j in best]

print(headline(articles[5]), "\n")
for score, j in similar_articles(5):
    print(f"   {score:.2f}  {headline(articles[j])}")
```

*A document can be used as a query. This is the mechanism behind "related articles" and behind many recommendation systems.*
</details>

### Exercise 4 (Keywords of a book)
In Notebook 03 the most frequent words of *Alice in Wonderland* were uninformative. Apply TF-IDF across **all** the books in `nltk.corpus.gutenberg` (each book is one document) and print the ten top-weighted words for `carroll-alice.txt` and for `melville-moby_dick.txt`.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
from nltk.corpus import gutenberg

book_ids = gutenberg.fileids()
books = [gutenberg.raw(book_id) for book_id in book_ids]

book_vectoriser = TfidfVectorizer(stop_words="english", sublinear_tf=True)
book_matrix = book_vectoriser.fit_transform(books)
book_words = book_vectoriser.get_feature_names_out()

for book_id in ["carroll-alice.txt", "melville-moby_dick.txt"]:
    row = book_matrix[book_ids.index(book_id)].toarray().ravel()
    best = row.argsort()[::-1][:10]
    print(book_id, [book_words[j] for j in best])
```

*Now the keywords are the words that belong to each book and to no other: the characters and creatures of Wonderland, and the vocabulary of whaling. The everyday words that topped the frequency list have been pushed down, because every book contains them.*
</details>

### Exercise 5 (How good is the search?)
The Reuters articles carry topic labels: `reuters.categories(file_id)` returns the list of topics of an article. Use them to **measure** the search engine. For the query `"wheat harvest"`, retrieve the top 20 articles and compute the share that are labelled `wheat` or `grain`. This measure is called *precision at 20*. Do the same for `"crude oil production"` with the label `crude`.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
def precision_at(query, accepted_labels, how_many=20):
    relevant = 0
    for score, i in search(query, how_many):
        if set(reuters.categories(file_ids[i])) & set(accepted_labels):
            relevant += 1
    return relevant / how_many

print("wheat harvest:       ", precision_at("wheat harvest", ["wheat", "grain"]))
print("crude oil production:", precision_at("crude oil production", ["crude"]))
```

*Looking at a few results and finding them reasonable is not an evaluation. With labelled data we can put a number on quality, and then compare two versions of the system objectively. Precision measures how many of the returned documents are relevant. Its companion, recall, measures how many of the relevant documents were returned.*
</details>

### Exercise 6 (An inverted index, a little harder)
Real search engines do not compare the query with every document. They keep an **inverted index**: a dictionary from each word to the set of documents that contain it. Build one for the Reuters articles (use `re.findall(r"[a-z]+", text.lower())` as the tokeniser), then write `containing_all(query)` that returns the ids of the documents containing **every** word of the query. How many articles mention both `coffee` and `brazil`?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
import re

index = {}
for i, article in enumerate(articles):
    for word in set(re.findall(r"[a-z]+", article.lower())):
        if word not in index:
            index[word] = set()
        index[word].add(i)

def containing_all(query):
    result = None
    for word in query.lower().split():
        found = index.get(word, set())
        result = found if result is None else result & found
    return result

matches = containing_all("coffee brazil")
print(len(matches), "articles")
for i in sorted(matches)[:5]:
    print("  ", headline(articles[i]))
```

*Looking up a word in a dictionary takes the same time whether the collection has ten thousand documents or ten billion. A search engine first uses the inverted index to find the small set of candidates, and only then ranks those candidates with a score such as TF-IDF.*
</details>

## 🔑 Key takeaways

- A **bag of words** turns a document into a vector of word counts. A collection becomes a **document-term matrix**.
- **Cosine similarity** compares the direction of two vectors and ignores their length, so long and short documents on the same topic match.
- **TF-IDF** weights a word by how often it occurs in a document and how rare it is across the collection. It down-weights uninformative words automatically.
- Document-term matrices are **sparse**. Store them sparsely, and never fit a vectoriser twice.
- A **search engine** is: vectorise the documents, vectorise the query the same way, rank by cosine similarity.
- Search quality can and should be **measured**, for example with precision.
- The bag of words ignores **order**, and treats every pair of different words as equally **unrelated**.

---
**Next:** *Notebook 09: Text Classification with Naive Bayes*, where the same word counts are used to make a decision: which category does this document belong to?

---
*© Ioannis Tsioulis. *From Words to Meaning: Natural Language Processing in Python*. Prepared for educational use.*